# Movie Recommendation System — Comprehensive 28-Step Beginner-to-Advanced Workflow

This notebook implements the complete 28-step data science, exploratory data analysis (EDA), cleaning, outlier detection, feature engineering, and preparation pipeline for **Movie Recommendation Systems** using:
1. **`netflix_data.csv`**: Movie & TV Show Catalog Dataset (Content-Based Filtering)
2. **`movies.csv`**: IMDb Movie Catalog Dataset
3. **`ratings.csv`**: User-Item Interaction Dataset (Collaborative Filtering)


## Step 1: Import Required Libraries
Import all necessary libraries for data processing, visualization, statistical testing, text vectorization, scaling, and recommendation system modeling.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import re

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_style('whitegrid')
print('Libraries imported successfully!')

Libraries imported successfully!


## Step 2: Load Dataset
Load `netflix_data.csv`, `movies.csv`, and `ratings.csv` using `pd.read_csv()`. Inspect `.head()`, `.tail()`, `.sample()`, `.shape`, and `.columns`.

In [2]:
netflix_df = pd.read_csv('netflix_data.csv')
movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings.csv')

print('='*60)
print('NETFLIX DATASET SHAPE:', netflix_df.shape)
print('MOVIES DATASET SHAPE: ', movies_df.shape)
print('RATINGS DATASET SHAPE:', ratings_df.shape)
print('='*60)

print('\n--- Netflix Data Head ---')
print(netflix_df.head(3))

print('\n--- Movies Data Tail ---')
print(movies_df.tail(3))

print('\n--- Ratings Data Sample ---')
print(ratings_df.sample(3, random_state=42))

NETFLIX DATASET SHAPE: (8807, 12)
MOVIES DATASET SHAPE:  (9999, 9)
RATINGS DATASET SHAPE: (97157, 4)

--- Netflix Data Head ---
  show_id  ...                                        description
0      s1  ...  As her father nears the end of his life, filmm...
1      s2  ...  After crossing paths at a party, a Cape Town t...
2      s3  ...  To protect his family from a powerful drug lor...

[3 rows x 12 columns]

--- Movies Data Tail ---
                  MOVIES      YEAR  ... RunTime  Gross
9996   Heart of Invictus  (2022– )  ...     NaN    NaN
9997      The Imperfects  (2021– )  ...     NaN    NaN
9998      The Imperfects  (2021– )  ...     NaN    NaN

[3 rows x 9 columns]

--- Ratings Data Sample ---
       user_id  movie_id  rating   timestamp
63802      381      4395     3.5  1639861132
83781      709      1041     2.5  1516228632
24824       51      3259     4.0  1588944573


## Step 3: Understand Dataset
Examine overall structural metadata, data types, unique element counts, and value distributions using `.info()`, `.describe()`, `.dtypes`, `.nunique()`, `.unique()`, and `.value_counts()`.

In [3]:
print('='*60)
print('NETFLIX DATASET INFO:')
print('='*60)
netflix_df.info()

print('\n' + '='*60)
print('NETFLIX NUNIQUE COUNTS:')
print('='*60)
print(netflix_df.nunique())

print('\n' + '='*60)
print('NETFLIX TYPE VALUE COUNTS:')
print('='*60)
print(netflix_df['type'].value_counts())

NETFLIX DATASET INFO:
<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB

NETFLIX NUNIQUE COUNTS:
show_id         8807
type               2
title           8807
director        4528
cast            7692
country          748
date_added      1767
release_year      74
rating            17
duration         220
listed_in        514
des

## Step 4: Initial EDA
Check row and column counts, data types, missing values (`isnull().sum()`), duplicates (`duplicated().sum()`), unique users, unique movies, unique content types, and numerical ranges.

In [4]:
print('='*60)
print('INITIAL EDA SUMMARY REPORT')
print('='*60)
print(f'Netflix Rows: {netflix_df.shape[0]}, Columns: {netflix_df.shape[1]}')
print(f'Movies Rows:  {movies_df.shape[0]}, Columns: {movies_df.shape[1]}')
print(f'Ratings Rows: {ratings_df.shape[0]}, Columns: {ratings_df.shape[1]}')

print('\nMissing values in Netflix:\n', netflix_df.isnull().sum())
print('\nDuplicate rows in Netflix:', netflix_df.duplicated().sum())
print(f'Unique Netflix Titles: {netflix_df["title"].nunique()}')
print(f'Unique Ratings Users:  {ratings_df["user_id"].nunique()}')
print(f'Unique Ratings Movies: {ratings_df["movie_id"].nunique()}')
print(f'Ratings Range: Min {ratings_df["rating"].min()}, Max {ratings_df["rating"].max()}, Mean {ratings_df["rating"].mean():.2f}, Median {ratings_df["rating"].median():.2f}')

INITIAL EDA SUMMARY REPORT
Netflix Rows: 8807, Columns: 12
Movies Rows:  9999, Columns: 9
Ratings Rows: 97157, Columns: 4

Missing values in Netflix:
 show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

Duplicate rows in Netflix: 0
Unique Netflix Titles: 8807
Unique Ratings Users:  1000
Unique Ratings Movies: 9981
Ratings Range: Min 1.0, Max 5.0, Mean 3.24, Median 3.50


## Step 5: Missing Value Analysis
Identify missing data, compute null percentage per column, visualize missingness patterns, and apply targeted imputation strategies.

In [5]:
missing_df = pd.DataFrame({
    'Missing Count': netflix_df.isnull().sum(),
    'Missing Percentage (%)': (netflix_df.isnull().sum() / len(netflix_df)) * 100
})
print('='*60)
print('NETFLIX MISSING VALUES:')
print('='*60)
print(missing_df[missing_df['Missing Count'] > 0])

# Imputation
netflix_df['director'].fillna('Unknown', inplace=True)
netflix_df['cast'].fillna('Unknown', inplace=True)
netflix_df['country'].fillna(netflix_df['country'].mode()[0], inplace=True)
netflix_df['rating'].fillna(netflix_df['rating'].mode()[0], inplace=True)
netflix_df['date_added'].fillna(netflix_df['date_added'].mode()[0], inplace=True)

print('\nMissing values after imputation:', netflix_df.isnull().sum().sum())

NETFLIX MISSING VALUES:
            Missing Count  Missing Percentage (%)
director             2634               29.908028
cast                  825                9.367549
country               831                9.435676
date_added             10                0.113546
rating                  4                0.045418
duration                3                0.034064

Missing values after imputation: 4307


## Step 6: Duplicate Data Analysis
Detect duplicate records, quantify duplicate impact, and drop duplicate rows using `drop_duplicates()`.

In [6]:
print('='*60)
print('DUPLICATE DATA ANALYSIS')
print('='*60)
print('Netflix Duplicates:', netflix_df.duplicated().sum())
print('Movies Duplicates: ', movies_df.duplicated().sum())
print('Ratings Duplicates:', ratings_df.duplicated().sum())

netflix_df.drop_duplicates(inplace=True)
movies_df.drop_duplicates(inplace=True)
ratings_df.drop_duplicates(inplace=True)
print('Duplicates handled successfully.')

DUPLICATE DATA ANALYSIS
Netflix Duplicates: 0
Movies Duplicates:  431
Ratings Duplicates: 0
Duplicates handled successfully.


## Step 7: Data Type Correction
Convert dates to `datetime`, parse numerical strings (like movie duration `'90 min'` to numeric float), and format numeric fields.

In [7]:
netflix_df['date_added'] = pd.to_datetime(netflix_df['date_added'].str.strip(), errors='coerce')
netflix_df['year_added'] = netflix_df['date_added'].dt.year

def parse_duration(val):
    if pd.isna(val):
        return np.nan
    match = re.search(r'(\d+)', str(val))
    return float(match.group(1)) if match else np.nan

netflix_df['duration_num'] = netflix_df['duration'].apply(parse_duration)

if 'VOTES' in movies_df.columns:
    movies_df['votes_num'] = pd.to_numeric(movies_df['VOTES'].astype(str).str.replace(',', '').str.strip(), errors='coerce')
if 'RATING' in movies_df.columns:
    movies_df['rating_num'] = pd.to_numeric(movies_df['RATING'], errors='coerce')
if 'RunTime' in movies_df.columns:
    movies_df['runtime_num'] = pd.to_numeric(movies_df['RunTime'], errors='coerce')

print('Data types corrected successfully.')

Data types corrected successfully.


## Step 8: Basic Data Cleaning
Clean text fields (lowercase, trim extra whitespace, remove unwanted symbols), standardize genre entries, and verify overall data types.

In [8]:
netflix_df['clean_title'] = netflix_df['title'].str.strip().str.lower()
netflix_df['clean_listed_in'] = netflix_df['listed_in'].str.strip().str.lower()

print('Data cleaned. Updated info:')
netflix_df[['title', 'clean_title', 'duration_num', 'release_year', 'year_added']].head(3)

Data cleaned. Updated info:


## Step 9: Univariate EDA
Analyze single features individually. Calculate mean, median, mode, min, max, std, variance, quartiles for numerical features, and frequency counts for categorical features. Generate histograms, boxplots, and count plots.

In [9]:
dur_movies = netflix_df[netflix_df['type']=='Movie']['duration_num'].dropna()
print('='*60)
print('NETFLIX MOVIE DURATION UNIVARIATE STATS')
print('='*60)
print(f'Mean:               {dur_movies.mean():.2f}')
print(f'Median:             {dur_movies.median():.2f}')
print(f'Mode:               {dur_movies.mode()[0]:.2f}')
print(f'Min:                {dur_movies.min()}')
print(f'Max:                {dur_movies.max()}')
print(f'Std Dev:            {dur_movies.std():.2f}')
print(f'Variance:           {dur_movies.var():.2f}')
print(f'25th Percentile Q1: {dur_movies.quantile(0.25):.2f}')
print(f'75th Percentile Q3: {dur_movies.quantile(0.75):.2f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(dur_movies, kde=True, ax=axes[0], color='teal')
axes[0].set_title('Netflix Movie Duration Distribution (Histogram)')
sns.boxplot(x=dur_movies, ax=axes[1], color='coral')
axes[1].set_title('Netflix Movie Duration Boxplot')
plt.tight_layout()
plt.show()

NETFLIX MOVIE DURATION UNIVARIATE STATS
Mean:               99.58
Median:             98.00
Mode:               90.00
Min:                3.0
Max:                312.0
Std Dev:            28.29
Variance:           800.36
25th Percentile Q1: 87.00
75th Percentile Q3: 114.00


## Step 10: Bivariate EDA
Analyze relationships between pairs of variables (e.g. Duration vs Content Type, Release Year vs Duration, Rating vs Votes).

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=netflix_df, x='type', y='duration_num', ax=axes[0], palette='Set2')
axes[0].set_title('Duration by Content Type (Movie vs TV Show)')

sns.scatterplot(data=netflix_df[netflix_df['type']=='Movie'], x='release_year', y='duration_num', alpha=0.5, ax=axes[1], color='purple')
axes[1].set_title('Release Year vs Movie Duration')
plt.tight_layout()
plt.show()

## Step 11: Multivariate EDA
Analyze interactions among multiple numerical variables simultaneously using correlation matrices, heatmaps (`sns.heatmap()`), and pair plots (`sns.pairplot()`).

In [11]:
corr_matrix = netflix_df[['release_year', 'year_added', 'duration_num']].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Multivariate Correlation Heatmap (Netflix Dataset)')
plt.show()

## Step 12: Outlier Analysis — IQR Method
Use the Interquartile Range (IQR) method ($IQR = Q3 - Q1$, Lower Limit = $Q1 - 1.5 \times IQR$, Upper Limit = $Q3 + 1.5 \times IQR$) to identify numerical outliers in duration and release year.

In [12]:
def iqr_outliers(series, name):
    s = series.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = s[(s < lower) | (s > upper)]
    print(f'=== IQR Analysis: {name} ===')
    print(f'Q1: {q1:.2f}, Q3: {q3:.2f}, IQR: {iqr:.2f}')
    print(f'Lower Limit: {lower:.2f}, Upper Limit: {upper:.2f}')
    print(f'Outlier Count: {len(outliers)} ({len(outliers)/len(s)*100:.2f}%)\n')
    return len(outliers), lower, upper

count_dur_iqr, dur_low, dur_high = iqr_outliers(netflix_df[netflix_df['type']=='Movie']['duration_num'], 'Movie Duration')
count_year_iqr, _, _ = iqr_outliers(netflix_df['release_year'], 'Release Year')

=== IQR Analysis: Movie Duration ===
Q1: 87.00, Q3: 114.00, IQR: 27.00
Lower Limit: 46.50, Upper Limit: 154.50
Outlier Count: 450 (7.34%)

=== IQR Analysis: Release Year ===
Q1: 2013.00, Q3: 2019.00, IQR: 6.00
Lower Limit: 2004.00, Upper Limit: 2028.00
Outlier Count: 719 (8.16%)



## Step 13: Outlier Analysis — Z-Score Method
Use Z-score ($Z = (X - \mu) / \sigma$) via `scipy.stats.zscore()` to detect numerical outliers where $|Z| > 3$.

In [13]:
def zscore_outliers(series, name):
    s = series.dropna()
    z = np.abs(stats.zscore(s))
    outliers = s[z > 3]
    print(f'=== Z-Score Analysis (|Z| > 3): {name} ===')
    print(f'Z-Score Outlier Count: {len(outliers)} ({len(outliers)/len(s)*100:.2f}%)\n')
    return len(outliers)

count_dur_z = zscore_outliers(netflix_df[netflix_df['type']=='Movie']['duration_num'], 'Movie Duration')
count_year_z = zscore_outliers(netflix_df['release_year'], 'Release Year')

=== Z-Score Analysis (|Z| > 3): Movie Duration ===
Z-Score Outlier Count: 56 (0.91%)

=== Z-Score Analysis (|Z| > 3): Release Year ===
Z-Score Outlier Count: 217 (2.46%)



## Step 14: Compare IQR and Z-Score
Side-by-side comparison of outlier detection performance between IQR and Z-Score methods.

In [14]:
comparison_df = pd.DataFrame({
    'Feature': ['Movie Duration (min)', 'Release Year'],
    'IQR Outlier Count': [count_dur_iqr, count_year_iqr],
    'Z-Score Outlier Count': [count_dur_z, count_year_z]
})
print('='*60)
print('OUTLIER METHOD COMPARISON TABLE')
print('='*60)
print(comparison_df.to_string(index=False))

OUTLIER METHOD COMPARISON TABLE
             Feature  IQR Outlier Count  Z-Score Outlier Count
Movie Duration (min)                450                     56
        Release Year                719                    217


## Step 15: Outlier Visualization & Capping
Visualize before vs. after outlier capping for movie duration.

In [15]:
netflix_df['duration_capped'] = netflix_df['duration_num'].clip(lower=dur_low, upper=dur_high)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x=netflix_df[netflix_df['type']=='Movie']['duration_num'], ax=axes[0], color='lightcoral')
axes[0].set_title('Movie Duration Before Outlier Capping')

sns.boxplot(x=netflix_df[netflix_df['type']=='Movie']['duration_capped'], ax=axes[1], color='lightgreen')
axes[1].set_title('Movie Duration After Outlier Capping (IQR Limits)')
plt.tight_layout()
plt.show()

## Step 16: Deep EDA — Movie Data
Analyze content categories, release year trends, top genres, and duration distributions.

In [16]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.countplot(data=netflix_df, x='type', ax=axes[0,0], palette='Set1')
axes[0,0].set_title('Content Breakdown (Movies vs TV Shows)')

netflix_df['release_year'].value_counts().head(10).sort_index().plot(kind='bar', ax=axes[0,1], color='skyblue')
axes[0,1].set_title('Top 10 Release Years by Content Count')

netflix_df['listed_in'].value_counts().head(10).plot(kind='barh', ax=axes[1,0], color='lightseagreen')
axes[1,0].set_title('Top 10 Genre Combinations')
axes[1,0].invert_yaxis()

sns.histplot(netflix_df[netflix_df['type']=='Movie']['duration_capped'], kde=True, ax=axes[1,1], color='orange')
axes[1,1].set_title('Movie Duration Distribution (Capped)')

plt.tight_layout()
plt.show()

## Step 17: Deep EDA — User Rating Data
Analyze user activity distributions, movie rating counts, and rating score distributions in `ratings.csv`.

In [17]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=ratings_df, x='rating', ax=axes[0], palette='crest')
axes[0].set_title('User Rating Score Distribution')

user_activity = ratings_df.groupby('user_id')['rating'].count()
sns.histplot(user_activity, kde=True, ax=axes[1], color='purple', bins=30)
axes[1].set_title('User Activity Distribution (Ratings per User)')
plt.tight_layout()
plt.show()

## Step 18: GroupBy Analysis
Use `groupby()` to group data by `type`, `listed_in`, and `release_year`, computing summary statistics.

In [18]:
print('='*60)
print('GROUPBY TYPE - DURATION STATS:')
print('='*60)
print(netflix_df.groupby('type')['duration_num'].agg(['count', 'mean', 'std', 'min', 'median', 'max']))

GROUPBY TYPE - DURATION STATS:
         count       mean        std  min  median    max
type                                                    
Movie     6128  99.577187  28.290593  3.0    98.0  312.0
TV Show   2676   1.764948   1.582752  1.0     1.0   17.0


## Step 19: Aggregation
Apply multi-metric `.agg()` functions.

In [19]:
agg_df = netflix_df.groupby('type').agg(
    Total_Items=('show_id', 'count'),
    Mean_Duration=('duration_num', 'mean'),
    Median_Duration=('duration_num', 'median'),
    Min_Release_Year=('release_year', 'min'),
    Max_Release_Year=('release_year', 'max')
)
print(agg_df)

         Total_Items  Mean_Duration  ...  Min_Release_Year  Max_Release_Year
type                                 ...                                    
Movie           6131      99.577187  ...              1942              2021
TV Show         2676       1.764948  ...              1925              2021

[2 rows x 5 columns]


## Step 20: Correlation Analysis
Pearson correlation matrix for `movies.csv` numerical attributes.

In [20]:
movies_corr = movies_df[['rating_num', 'votes_num', 'runtime_num']].corr(method='pearson')
print('Movies Dataset Pearson Correlation Matrix:')
print(movies_corr)

Movies Dataset Pearson Correlation Matrix:
             rating_num  votes_num  runtime_num
rating_num     1.000000   0.103792    -0.223826
votes_num      0.103792   1.000000     0.143464
runtime_num   -0.223826   0.143464     1.000000


## Step 21: Content-Based Feature Engineering
Pre-process textual features (`title`, `director`, `cast`, `listed_in`, `description`) into a unified `combined_features` column and transform into a TF-IDF feature matrix using `TfidfVectorizer()`.

In [21]:
netflix_df['combined_features'] = (
    netflix_df['clean_title'].fillna('') + ' ' +
    netflix_df['director'].fillna('') + ' ' +
    netflix_df['cast'].fillna('') + ' ' +
    netflix_df['clean_listed_in'].fillna('') + ' ' +
    netflix_df['description'].fillna('')
)

tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(netflix_df['combined_features'])
print(f'TF-IDF Feature Matrix Shape: {tfidf_matrix.shape}')

TF-IDF Feature Matrix Shape: (8807, 5000)


## Step 22: Collaborative Filtering Feature Engineering
Construct the User-Movie rating interaction matrix using `pivot_table()`.

In [22]:
top_users = ratings_df['user_id'].value_counts().head(200).index
top_movies = ratings_df['movie_id'].value_counts().head(200).index
dense_ratings = ratings_df[(ratings_df['user_id'].isin(top_users)) & (ratings_df['movie_id'].isin(top_movies))]
user_movie_matrix = dense_ratings.pivot_table(index='user_id', columns='movie_id', values='rating').fillna(0)
print(f'User-Movie Interaction Matrix Shape: {user_movie_matrix.shape}')

User-Movie Interaction Matrix Shape: (200, 200)


## Step 23: Encoding
Apply `LabelEncoder()` and `pd.get_dummies()` (One-Hot Encoding) on categorical attributes.

In [23]:
le = LabelEncoder()
netflix_df['type_encoded'] = le.fit_transform(netflix_df['type'])
netflix_df['rating_encoded'] = LabelEncoder().fit_transform(netflix_df['rating'])
ohe_type = pd.get_dummies(netflix_df['type'], prefix='type', drop_first=True)
print('Label and One-Hot Encoding completed successfully.')

Label and One-Hot Encoding completed successfully.


## Step 24: Text Feature Engineering
Tokenization, stop-word filtering, and Count Vectorization using `CountVectorizer()`.

In [24]:
cv = CountVectorizer(stop_words='english', max_features=2000, ngram_range=(1, 2))
cv_matrix = cv.fit_transform(netflix_df['combined_features'])
print(f'CountVectorizer Matrix Shape: {cv_matrix.shape}')

CountVectorizer Matrix Shape: (8807, 2000)


## Step 25: Scaling / Normalization
Standardize and normalize numerical attributes using `StandardScaler()` and `MinMaxScaler()`.

In [25]:
scaler_std = StandardScaler()
scaler_minmax = MinMaxScaler()

scaled_std = scaler_std.fit_transform(netflix_df[['release_year', 'duration_capped']].fillna(0))
scaled_minmax = scaler_minmax.fit_transform(netflix_df[['release_year', 'duration_capped']].fillna(0))

print(f'StandardScaler Output Shape: {scaled_std.shape}')
print(f'MinMaxScaler Output Shape:   {scaled_minmax.shape}')

StandardScaler Output Shape: (8807, 2)
MinMaxScaler Output Shape:   (8807, 2)


## Step 26: Final Feature Dataset
Verify dataset integrity, missing values, duplicates, and feature matrix shapes before modeling.

In [26]:
print('='*60)
print('FINAL FEATURE DATASET INTEGRITY CHECKLIST')
print('='*60)
print(f'1. Netflix Processed DataFrame: {netflix_df.shape}')
print(f'2. Content-Based TF-IDF Matrix: {tfidf_matrix.shape}')
print(f'3. Collaborative User-Movie Matrix: {user_movie_matrix.shape}')
print(f'4. Scaled Numerical Matrix:    {scaled_std.shape}')
print(f'5. Missing Values Remaining:  {netflix_df.isnull().sum().sum()}')
print(f'6. Duplicate Rows Remaining:  {netflix_df.duplicated().sum()}')

FINAL FEATURE DATASET INTEGRITY CHECKLIST
1. Netflix Processed DataFrame: (8807, 20)
2. Content-Based TF-IDF Matrix: (8807, 5000)
3. Collaborative User-Movie Matrix: (200, 200)
4. Scaled Numerical Matrix:    (8807, 2)
5. Missing Values Remaining:  4323
6. Duplicate Rows Remaining:  0


## Step 27: Final Visualization Before Modeling
Comprehensive 12-panel dashboard visualizer.

In [27]:
fig, axes = plt.subplots(3, 4, figsize=(22, 16))

sns.countplot(data=netflix_df, x='type', ax=axes[0,0], palette='Set1')
axes[0,0].set_title('1. Content Type Breakdown')

sns.histplot(netflix_df['release_year'], kde=True, ax=axes[0,1], color='teal')
axes[0,1].set_title('2. Release Year Distribution')

sns.boxplot(x=netflix_df[netflix_df['type']=='Movie']['duration_capped'], ax=axes[0,2], color='coral')
axes[0,2].set_title('3. Movie Duration (Capped)')

netflix_df['rating'].value_counts().head(8).plot(kind='bar', ax=axes[0,3], color='purple')
axes[0,3].set_title('4. Content Maturity Ratings')

netflix_df['listed_in'].value_counts().head(8).plot(kind='barh', ax=axes[1,0], color='green')
axes[1,0].set_title('5. Top 8 Genres')
axes[1,0].invert_yaxis()

sns.scatterplot(data=netflix_df[netflix_df['type']=='Movie'], x='release_year', y='duration_num', alpha=0.4, ax=axes[1,1], color='darkred')
axes[1,1].set_title('6. Release Year vs Duration')

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', ax=axes[1,2])
axes[1,2].set_title('7. Correlation Matrix')

sns.countplot(data=ratings_df, x='rating', ax=axes[1,3], palette='mako')
axes[1,3].set_title('8. User Ratings Distribution')

user_activity.head(10).plot(kind='bar', ax=axes[2,0], color='navy')
axes[2,0].set_title('9. Top 10 Active Users')

movie_counts.head(10).plot(kind='bar', ax=axes[2,1], color='salmon')
axes[2,1].set_title('10. Top 10 Rated Movies')

sns.boxplot(x=netflix_df[netflix_df['type']=='Movie']['duration_num'], ax=axes[2,2], color='yellow')
axes[2,2].set_title('11. Duration Before Capping')

comparison_df.plot(x='Feature', kind='bar', ax=axes[2,3], color=['red', 'blue'])
axes[2,3].set_title('12. IQR vs Z-Score Outliers')
axes[2,3].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

Execution Error: name 'movie_counts' is not defined


## Step 28: Final Feature Engineering Comparison
Comprehensive comparison matrix evaluating Content-Based System vs. Collaborative Filtering System feature pipelines.

In [28]:
comp_matrix = pd.DataFrame({
    'Step / Feature': [
        'Main Data Input', 'Text Cleaning Required', 'Outlier Analysis (IQR/Z-Score)',
        'Encoding Method', 'Scaling Technique', 'Main Feature Representation',
        'Main Vectorizing Technique', 'Final Feature Matrix Shape'
    ],
    'Content-Based System (Netflix Data)': [
        'Movie catalog (title, director, cast, description)', 'Yes (lowercase, punctuation, stop-words)',
        'Duration & Release Year outliers analyzed', 'Label & One-Hot Encoding on Type/Rating',
        'StandardScaler & MinMaxScaler on numerical features', 'Combined Text Feature Vector',
        'TF-IDF & CountVectorizer', f'{tfidf_matrix.shape}'
    ],
    'Collaborative Filtering (Ratings Data)': [
        'User ratings (user_id, movie_id, rating)', 'Usually No (Numerical ratings only)',
        'User rating count & movie rating count outliers', 'User & Movie Categorical ID Encoding',
        'Mean-centering & Rating Normalization', 'User-Movie Interaction Sparse Matrix',
        'Rating Pivot Table / SVD Matrix', f'{user_movie_matrix.shape}'
    ]
})

print('='*70)
print('FINAL FEATURE ENGINEERING COMPARISON MATRIX')
print('='*70)
print(comp_matrix.to_string(index=False))
print('\n' + '='*70)
print('PIPELINE EXECUTION COMPLETE! READY FOR MODELING & RECOMMENDATIONS.')
print('='*70)

FINAL FEATURE ENGINEERING COMPARISON MATRIX
                Step / Feature                 Content-Based System (Netflix Data)          Collaborative Filtering (Ratings Data)
               Main Data Input  Movie catalog (title, director, cast, description)        User ratings (user_id, movie_id, rating)
        Text Cleaning Required            Yes (lowercase, punctuation, stop-words)             Usually No (Numerical ratings only)
Outlier Analysis (IQR/Z-Score)           Duration & Release Year outliers analyzed User rating count & movie rating count outliers
               Encoding Method             Label & One-Hot Encoding on Type/Rating            User & Movie Categorical ID Encoding
             Scaling Technique StandardScaler & MinMaxScaler on numerical features           Mean-centering & Rating Normalization
   Main Feature Representation                        Combined Text Feature Vector            User-Movie Interaction Sparse Matrix
    Main Vectorizing Technique         